# Kafka Architecture: Brokers, Controllers & KRaft

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**This topic runs no Spark job.** The driver container this Jupyter kernel runs in has no Docker CLI/socket access, so the CLI steps below (marked "host terminal") must be run from your own machine's terminal, not from a notebook cell -- exactly the same constraint `kafka-replication-fault-tolerance` (later in this curriculum) documents for its broker-kill exercise. The Python cells below talk to the brokers directly over the network (`kafka-1:9092` / `kafka-2:9092` / `kafka-3:9092`, reachable in-cluster), which needs no Docker access at all.

In [ ]:
from kafka import KafkaProducer

BOOTSTRAP_SERVERS = ["kafka-1:9092", "kafka-2:9092", "kafka-3:9092"]

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
print("Connected to the KRaft cluster:", producer.bootstrap_connected())

## 1. Confirm the KRaft quorum (host terminal)

Open a terminal on your own machine (not this notebook) and run:

```
docker ps --filter "name=spark-kafka" --format "table {{.Names}}\t{{.Image}}\t{{.Status}}"
```

You should see 3 `spark-kafka-N` containers and, checking full `docker ps` output, no ZooKeeper container anywhere -- these 3 processes are the entire cluster, control plane included.

Then run:

```
docker exec spark-kafka-1 /opt/kafka/bin/kafka-metadata-quorum.sh --bootstrap-server localhost:9092 describe --status
```

Look at `CurrentVoters` (should list all 3 broker node IDs -- the KRaft quorum) and `LeaderId` (the active controller among them). See `concept.md` for what replaced ZooKeeper here.

In [ ]:
# Convenience copy-paste command for the step above -- adjust the broker
# suffix (spark-kafka-1) if you killed/respawned a different one.
print("docker exec spark-kafka-1 /opt/kafka/bin/kafka-metadata-quorum.sh "
      "--bootstrap-server localhost:9092 describe --status")

## 2. Create a topic and produce a few messages (kafka-python, in-cluster)

Kafka's broker auto-creates a topic the first time something produces to it (`KAFKA_AUTO_CREATE_TOPICS_ENABLE=true` on this cluster) -- no `KafkaAdminClient` needed (it doesn't work reliably against this combined broker+controller KRaft setup, a documented limitation this repo's tooling already works around).

In [ ]:
TOPIC = "kraft-demo-topic"

for i in range(6):
    future = producer.send(TOPIC, key=f"key-{i % 3}".encode(), value=f"event-{i}".encode())
    metadata = future.get(timeout=10)
    print(f"sent event-{i} -> partition {metadata.partition}, offset {metadata.offset}")

producer.flush()

## 3. Describe the topic: partitions, replication, leader, ISR (host terminal)

In your host terminal, run:

```
docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic kraft-demo-topic
```

The first line (`Topic: ... PartitionCount: ... ReplicationFactor: ...`) is the topic-level summary. Each indented line after it is one partition:

- **`Leader`** -- the broker ID currently serving reads/writes for that partition.
- **`Replicas`** -- every broker ID that holds a copy of that partition (should be all 3, since RF=3 was requested).
- **`Isr`** -- the subset of `Replicas` that are fully caught up right now. With nothing failed yet, `Isr` should equal `Replicas`.

In [ ]:
print("docker exec spark-kafka-1 /opt/kafka/bin/kafka-topics.sh "
      "--bootstrap-server localhost:9092 --describe --topic kraft-demo-topic")

## 4. KRaft vs ZooKeeper -- what actually changed

You've now seen the two pieces of evidence that matter: `docker ps` shows only broker processes (no separate coordination service), and `kafka-metadata-quorum.sh` shows those same broker processes voting on cluster metadata via Raft. Read `concept.md`'s "Why it matters" section for the full ZooKeeper-era contrast -- what it used to hold, why brokers watched it, and why removing it was worth doing.

The rest of this curriculum builds on the two vocabulary terms this topic introduced -- **replication factor** and **ISR** -- most directly in `kafka-replication-fault-tolerance`, where you'll watch a real leader election and ISR shrink/grow happen live.